# GLiNER2 Jetson API — Worked Examples

Runnable client examples for the self-hosted GLiNER2 service on `jarvita-agx`
(NVIDIA Jetson AGX Orin 64GB).

Covered here:

1. Setup and connectivity
2. `GET /health` and `GET /version`
3. `POST /extract_entities`
4. `POST /classify_text`
5. `POST /extract_structured`
6. `POST /extract_multitask`
7. Error semantics (400 / 413 / 503 / 504)
8. Throughput: sequential vs concurrent, plus reference batching numbers
9. Relation extraction (GLiNER2.5)

Companion docs: [`index.md`](index.md), [`api.md`](api.md),
[`runbook.md`](runbook.md), [`wiki.md`](wiki.md).

Only dependency is `requests`:

```bash
pip install requests
```

## 1. Setup

`BASE_URL` is read from the `GLINER_BASE_URL` environment variable so the same
notebook works against the Jetson, a local dev server, or a port-forward. Change
the fallback if your deployment differs.

| Deployment | Base URL |
|---|---|
| `jarvita-agx` on the LAN | `http://192.168.1.177:8013` |
| Container on the local host | `http://localhost:8013` |
| `make run` / `make dev` locally | `http://localhost:8125` |

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor

import requests

# Point this at your deployment. Container port 8012 is published on host 8013.
BASE_URL = os.environ.get("GLINER_BASE_URL", "http://192.168.1.177:8013")

# Inference is bounded server-side by REQUEST_TIMEOUT_SECONDS (default 120),
# so a client timeout a bit above that is the sane default.
TIMEOUT = 130

session = requests.Session()
session.headers.update({"Content-Type": "application/json"})


def get(path):
    """GET a path and return parsed JSON."""
    r = session.get(f"{BASE_URL}{path}", timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post(path, payload):
    """POST JSON and return parsed JSON. Raises on non-2xx."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()


def post_raw(path, payload):
    """POST JSON and return (status_code, parsed_body_or_text). Never raises."""
    r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text


def show(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False))


print("BASE_URL =", BASE_URL)

## 2. Health and version

`GET /health` is a liveness check plus model state. It never touches the GPU, so
it stays responsive while an inference is in flight — which is exactly why it is
the right thing for a container healthcheck.

Fields to read:

- `loaded` — is the model resident? With `MODEL_PRELOAD=0` this stays `false`
  until the first inference request.
- `device` — `"cuda"` on a healthy Jetson. `"cpu"` means the GPU was not visible
  to the container; see the runbook.
- `architecture` / `model_class` — present on builds with GLiNER2.5 dispatch.
  `span` + `GLiNER2` for the 2.x models, `boundary` + `AutoExtractor` for 2.5.

In [ ]:
health = get("/health")
show(health)

assert health["status"] == "ok"
if health.get("device") != "cuda":
    print("\nWARNING: not running on GPU. On a Jetson this is a fault, not a mode.")

In [ ]:
show(get("/version"))

`/version` reports whatever `gliner2.__version__` says for the pin baked into
the image. Check it first when two containers behave differently.

`/health` is *liveness*, not proof the model works. A real forward pass is the
only readiness check that means anything:

In [ ]:
smoke = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time"],
})
show(smoke)

# Verified response from jarvita-agx:
# {"entities": {"medication": ["ibuprofen"], "dosage": ["400mg"],
#               "symptom": ["headache"], "time": ["2 PM"]}}

## 3. Entity extraction — `POST /extract_entities`

Zero-shot NER. The label set travels with the request; nothing is fine-tuned,
and the same loaded weights answer any label set you send.

| Field | Type | Notes |
|---|---|---|
| `text` | string | non-empty, `<= MAX_TEXT_CHARS` (default 20000) |
| `labels` | list of strings, **or** object label → description | non-empty, `<= MAX_LABELS` (default 256) |

The response keys are exactly the labels you sent. A label with no match maps to
an empty list.

In [ ]:
clinical = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": ["medication", "dosage", "symptom", "time"],
})
show(clinical)

The same model, a completely different domain and label set — no reload, no
fine-tune. This is the whole point of the zero-shot schema:

In [ ]:
business = post("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location", "job_title"],
})
show(business)

### Steering labels with descriptions

Passing `labels` as an object maps each label to a natural-language description.
Use this when a bare label is ambiguous — `"reference"` could mean a citation, a
ticket number, or a job reference, and the description settles it.

In [ ]:
described = post("/extract_entities", {
    "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
    "labels": {
        "medication": "name of a drug administered to the patient",
        "dosage": "amount and unit of the drug",
    },
})
show(described)

### Multilingual

`fastino/gliner2.5-multi-v1` is the multilingual checkpoint (mDeBERTa-v3-base
encoder). The cell below only produces meaningful results if the service is
running that `MODEL_ID` — the English-only models will do something, but not
something you should rely on.

In [ ]:
current_model = get("/health")["model_id"]
print("serving:", current_model)

if "multi" not in current_model:
    print("NOTE: not a multilingual checkpoint — treat the output below as a curiosity.")

show(post("/extract_entities", {
    "text": "Luca de Meo, PDG de Renault, a annoncé une nouvelle usine à Douai.",
    "labels": ["person", "company", "location"],
}))

Worth knowing: on a four-sentence comparison run on this box, `gliner2.5-multi`
returned `"PDG de Renault"` as the company for a sentence like this one, where
`gliner2-large` returned `"Renault"` — the 2.5 model swept the job title into
the span. **Four sentences is not an evaluation.** It is enough to justify one
habit: when you change `MODEL_ID`, re-check span boundaries against your own
texts before assuming downstream string matching still works.

## 4. Classification — `POST /classify_text`

| Field | Type | Notes |
|---|---|---|
| `text` | string | non-empty, `<= MAX_TEXT_CHARS` |
| `labels` | list of class labels, **or** object task name → class labels | non-empty, `<= MAX_LABELS` |

Prefer the object form. It names the task, so the response key is predictable
instead of something you have to discover.

In [ ]:
sentiment = post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {"sentiment": ["positive", "negative", "neutral"]},
})
show(sentiment)

# Verified response from jarvita-agx: {"sentiment": "negative"}

Several independent classification tasks can ride along in one request — add
more keys to the `labels` object. One forward pass, several answers.

In [ ]:
show(post("/classify_text", {
    "text": "The battery life is terrible and it overheats constantly.",
    "labels": {
        "sentiment": ["positive", "negative", "neutral"],
        "topic": ["battery", "display", "software", "build quality"],
    },
}))

## 5. Structured extraction — `POST /extract_structured`

Declare a record schema, get records filled from free text.

Each top-level key of `schema` is a record name. Its value is a list of
`::`-delimited field specs:

```
"<field_name>::<dtype>"
"<field_name>::<dtype>::<description>"
```

`dtype` is `str` or `list`. The optional third segment steers extraction.

In [ ]:
product = post("/extract_structured", {
    "text": "The Sony WH-1000XM5 headphones cost $399 and ship in 3 days.",
    "schema": {"product": ["name::str", "price::str", "shipping::str"]},
})
show(product)

# Verified response from jarvita-agx:
# {"product": [{"name": "Sony WH-1000XM5 headphones",
#               "price": "$399", "shipping": "3 days"}]}

Note the value is a **list** of records, because a schema can match more than
once in a document. Only index `[0]` when you know the text holds a single
record.

In [ ]:
records = product["product"]
print(f"{len(records)} record(s)")
for i, rec in enumerate(records):
    print(f"  [{i}] {rec}")

With field descriptions, for a schema where bare names would be ambiguous:

In [ ]:
show(post("/extract_structured", {
    "text": "Goldman Sachs processed a $2.5M equity trade for Tesla Inc.",
    "schema": {
        "transaction": [
            "broker::str::Financial institution",
            "amount::str::Transaction amount",
            "security::str::Stock name",
        ]
    },
}))

## 6. Multi-task — `POST /extract_multitask`

Entities, a classification, and a structured record from **one** forward pass
over one text.

**Everything nests under `schema_config`.** There are no top-level `entities` /
`classification` / `structure` keys. Putting them at the top level is the most
common mistake with this endpoint and returns `400`.

| Field | Notes |
|---|---|
| `schema_config.entities` | list of entity labels |
| `schema_config.classification` | `{"name": str, "labels": [str, ...]}` |
| `schema_config.structure` | `{"name": str, "fields": [{"name", "dtype", "description", "choices"}, ...]}` |

All three sub-keys are optional; supply at least one.

In [ ]:
multitask = post("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "schema_config": {
        "entities": ["company", "person", "location"],
        "classification": {"name": "sentiment", "labels": ["positive", "negative"]},
        "structure": {
            "name": "announcement",
            "fields": [
                {"name": "who", "dtype": "str"},
                {"name": "what", "dtype": "str"},
            ],
        },
    },
})
show(multitask)

# Verified response from jarvita-agx:
# {"announcement": [{"who": "Tim Cook", "what": "record revenue"}],
#  "entities": {"company": ["Apple"], "person": ["Tim Cook"],
#               "location": ["Cupertino"]},
#  "sentiment": "positive"}

The response is flat: entities under `entities`, the classification under the
`name` you gave it, the structure under its `name` as a list of records. Key
ordering is not guaranteed — always address by key.

In [ ]:
print("entities      :", multitask["entities"])
print("sentiment     :", multitask["sentiment"])
print("announcement  :", multitask["announcement"][0])

### The wrong shape, for reference

This is what a top-level payload gets you. Run it once so the failure is
recognizable when it shows up in a client:

In [ ]:
status, body = post_raw("/extract_multitask", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "entities": ["company", "person", "location"],   # WRONG: must nest under schema_config
})
print("HTTP", status)
show(body)

### `choices`: constraining a field to a closed vocabulary

`structure.fields` entries accept `choices`, which restricts the field to a
fixed set rather than free extraction. Useful when the downstream consumer
expects an enum.

In [ ]:
show(post("/extract_multitask", {
    "text": "Support ticket: the checkout page returns a 500 error for all EU customers. "
            "This is blocking revenue and needs attention today.",
    "schema_config": {
        "entities": ["component", "error_code", "region"],
        "classification": {"name": "urgency", "labels": ["low", "medium", "high"]},
        "structure": {
            "name": "ticket",
            "fields": [
                {"name": "summary", "dtype": "str", "description": "one-line problem statement"},
                {"name": "area", "dtype": "str", "choices": ["frontend", "backend", "infrastructure"]},
            ],
        },
    },
}))

## 7. Error semantics

The service bounds everything and returns defined status codes. These are not
edge cases to ignore — `503` in particular is normal operation on a single-GPU
box.

| Status | Trigger |
|---|---|
| 400 | Malformed payload: missing/empty `text`, bad `labels` type, missing `schema_config`, or a limit on label/field count |
| 413 | `text` longer than `MAX_TEXT_CHARS` (default 20000) |
| 503 | No inference slot free within `INFERENCE_ACQUIRE_TIMEOUT_SECONDS` (default 10) |
| 504 | Inference exceeded `REQUEST_TIMEOUT_SECONDS` (default 120) |
| 500 | Exception inside the model |

In [ ]:
# 400 - missing text
print("--- missing 'text' ---")
status, body = post_raw("/extract_entities", {"labels": ["person"]})
print("HTTP", status, "|", body)

# 400 - labels of the wrong type
print("\n--- 'labels' as a string ---")
status, body = post_raw("/extract_entities", {"text": "Tim Cook.", "labels": "person"})
print("HTTP", status, "|", body)

# 413 - text over MAX_TEXT_CHARS
print("\n--- oversized 'text' ---")
max_chars = 20_000  # server default; raise here if your deployment overrides it
status, body = post_raw("/extract_entities", {"text": "a" * (max_chars + 1), "labels": ["x"]})
print("HTTP", status, "|", body)

### Handling 503 correctly

`503` is backpressure. The server is telling you the GPU queue is full, which on
a box with `MAX_CONCURRENT_INFERENCES=1` is the designed behavior under load.
The correct client response is retry with backoff and jitter — **not** more
concurrent requests, which makes it worse.

In [ ]:
import random


def post_with_retry(path, payload, attempts=5, base_delay=0.5):
    """POST with exponential backoff + jitter on 503 (busy) and 504 (timeout)."""
    for attempt in range(attempts):
        r = session.post(f"{BASE_URL}{path}", json=payload, timeout=TIMEOUT)
        if r.status_code not in (503, 504):
            r.raise_for_status()
            return r.json()
        if attempt == attempts - 1:
            r.raise_for_status()
        delay = base_delay * (2 ** attempt) + random.uniform(0, base_delay)
        print(f"  HTTP {r.status_code}, retrying in {delay:.2f}s "
              f"(attempt {attempt + 1}/{attempts})")
        time.sleep(delay)


show(post_with_retry("/extract_entities", {
    "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
    "labels": ["company", "person", "location"],
}))

## 8. Throughput: sequential vs concurrent

Two things get measured below, and they are not the same thing:

- **Sequential HTTP** — 16 requests, one after another. This is what a naive
  bulk client does.
- **Concurrent HTTP** — 16 requests fired from a thread pool. With
  `MAX_CONCURRENT_INFERENCES=1` the server serializes them at the semaphore
  anyway, so this mostly measures how well overlapping request handling hides
  HTTP overhead — and it will start returning `503` once requests queue past
  `INFERENCE_ACQUIRE_TIMEOUT_SECONDS`.

Timings depend on the model, the document, and whatever else is running on the
box. Run this on your own deployment rather than trusting a number in a doc.

In [ ]:
DOC = "Apple CEO Tim Cook announced record revenue in Cupertino."
LABELS = ["company", "person", "location"]
N = 16

payload = {"text": DOC, "labels": LABELS}

# Warm up: with MODEL_PRELOAD=0 the first request pays the model load cost and
# would otherwise dominate the measurement.
post("/extract_entities", payload)
print("warm-up done")

In [ ]:
# --- Sequential ---
t0 = time.perf_counter()
for _ in range(N):
    post("/extract_entities", payload)
seq_total_ms = (time.perf_counter() - t0) * 1000

print(f"sequential : {seq_total_ms:8.1f} ms total   "
      f"{seq_total_ms / N:6.1f} ms/doc")

In [ ]:
# --- Concurrent over HTTP ---
def one_call(_):
    r = session.post(f"{BASE_URL}/extract_entities", json=payload, timeout=TIMEOUT)
    return r.status_code


t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=8) as pool:
    statuses = list(pool.map(one_call, range(N)))
conc_total_ms = (time.perf_counter() - t0) * 1000

ok = sum(1 for s in statuses if s == 200)
busy = sum(1 for s in statuses if s == 503)

print(f"concurrent : {conc_total_ms:8.1f} ms total   "
      f"{conc_total_ms / N:6.1f} ms/doc")
print(f"             {ok} x 200, {busy} x 503 (busy)")
print(f"\nspeedup vs sequential: {seq_total_ms / conc_total_ms:.2f}x")

if busy:
    print("\n503s are expected here: the server has one inference slot and rejects "
          "anything that cannot get it within INFERENCE_ACQUIRE_TIMEOUT_SECONDS.")

### Reference: true model-level batching

The numbers above are HTTP-level. Separately, **model-level batching** was
measured directly against `gliner2.5-multi` on `jarvita-agx` — 16 identical
documents, one batched forward pass versus 16 sequential ones:

| Mode | Total | Per document |
|---|---|---|
| Batched | 384 ms | 24.0 ms |
| Sequential | 1607 ms | 100.4 ms |

**4.2x speedup per document.** Measured with another LLM sharing the box, so
treat it as indicative of this deployment rather than a rigorous benchmark.

The catch: **the HTTP API takes one `text` per request**, so that 4.2x is not
reachable from a client today. Getting it means adding a batch endpoint to
`app.py` or using the `gliner2` library in-process. On a single-GPU box this is
a far better throughput lever than raising `MAX_CONCURRENT_INFERENCES`, which
just moves the queue without making the GPU faster.

For context, single-document latency measured on the same box (average of 5
runs, shared box, indicative only):

| Test | `gliner2.5-multi` | `gliner2-large` |
|---|---|---|
| clinical entity extraction | 117.6 ms | 171.9 ms |
| business entity extraction | 172.0 ms | 149.5 ms |
| spanish entity extraction | 103.1 ms | 170.8 ms |
| french entity extraction | 107.9 ms | 164.8 ms |
| classification | 79.6 ms | 111.2 ms |

In [ ]:
# Per-endpoint latency on your deployment, for comparison with the table above.
cases = [
    ("extract_entities  (clinical)", "/extract_entities", {
        "text": "Patient received 400mg ibuprofen for severe headache at 2 PM.",
        "labels": ["medication", "dosage", "symptom", "time"],
    }),
    ("extract_entities  (business)", "/extract_entities", {
        "text": "Apple CEO Tim Cook announced record revenue in Cupertino.",
        "labels": ["company", "person", "location"],
    }),
    ("classify_text", "/classify_text", {
        "text": "The battery life is terrible and it overheats constantly.",
        "labels": {"sentiment": ["positive", "negative", "neutral"]},
    }),
    ("extract_structured", "/extract_structured", {
        "text": "The Sony WH-1000XM5 headphones cost $399 and ship in 3 days.",
        "schema": {"product": ["name::str", "price::str", "shipping::str"]},
    }),
]

RUNS = 5
print(f"{'endpoint':<32} {'mean ms':>9} {'min ms':>9} {'max ms':>9}")
print("-" * 62)
for name, path, body in cases:
    post(path, body)  # warm
    samples = []
    for _ in range(RUNS):
        t = time.perf_counter()
        post(path, body)
        samples.append((time.perf_counter() - t) * 1000)
    print(f"{name:<32} {sum(samples)/len(samples):9.1f} "
          f"{min(samples):9.1f} {max(samples):9.1f}")

print(f"\nmodel: {get('/health')['model_id']}")
print("Shared box - these are indicative, not a controlled benchmark.")

## 9. Relation extraction (GLiNER2.5)

GLiNER2.5 models can extract **typed relations**: given a text and a set of
relation names, they return pairs.

Verified directly against `gliner2.5-multi-v1` on `jarvita-agx`:

> Satya Nadella, CEO of Microsoft, met Sam Altman of OpenAI in Seattle to
> discuss the Azure partnership.

with relations `["works_for", "met_with", "located_in"]`:

```json
{
  "relation_extraction": {
    "works_for": [["Satya Nadella", "Microsoft"]],
    "met_with": [["Satya Nadella", "Sam Altman"]],
    "located_in": [["Sam Altman", "Seattle"], ["OpenAI", "Seattle"]]
  }
}
```

Each relation maps to a list of `[subject, object]` pairs. Note `located_in`
returning two pairs — one person and one organization. The model extracts what
the sentence supports; it does not apply a type constraint you did not give it.

**This is a model capability, not an endpoint on this service.** `app.py`'s
multi-task handler builds a schema from `entities`, `classification`, and
`structure` only. The cell below demonstrates that empirically — an unknown
`relations` key in `schema_config` is accepted by validation and then silently
ignored during schema construction.

In [ ]:
RELATION_TEXT = (
    "Satya Nadella, CEO of Microsoft, met Sam Altman of OpenAI in Seattle "
    "to discuss the Azure partnership."
)
RELATIONS = ["works_for", "met_with", "located_in"]

status, body = post_raw("/extract_multitask", {
    "text": RELATION_TEXT,
    "schema_config": {
        "entities": ["person", "company", "location"],
        "relations": RELATIONS,   # not handled by app.py - expect it to be dropped
    },
})

print("HTTP", status)
show(body)

if isinstance(body, dict) and "relation_extraction" not in body:
    print("\nConfirmed: 'relations' was ignored. Relation extraction is not "
          "exposed by this API build.")

### What the reference output looks like

Kept here so downstream parsing can be written and tested against the real
shape before an endpoint exists:

In [ ]:
# Captured from gliner2.5-multi-v1 on jarvita-agx, via the library directly.
REFERENCE_RELATIONS = {
    "relation_extraction": {
        "works_for": [["Satya Nadella", "Microsoft"]],
        "met_with": [["Satya Nadella", "Sam Altman"]],
        "located_in": [["Sam Altman", "Seattle"], ["OpenAI", "Seattle"]],
    }
}

show(REFERENCE_RELATIONS)

print("\nflattened as triples:")
for rel, pairs in REFERENCE_RELATIONS["relation_extraction"].items():
    for subject, obj in pairs:
        print(f"  ({subject}) --[{rel}]--> ({obj})")

### Getting relations over HTTP

Two routes:

1. **Add an endpoint.** Extend `app.py` — either a dedicated route or a
   `relations` branch in the `/extract_multitask` schema builder — and route it
   through the existing `_run_inference` wrapper so it inherits the semaphore,
   the timeout, and the timing logs. Keep the existing payload contracts
   backward compatible (see [`../AGENTS.md`](../AGENTS.md)).
2. **Use the library in-process** on the Jetson, where batching is also
   available.

Either way it requires a GLiNER2.5 (`boundary`) checkpoint. Confirm what is
loaded before assuming relations are even possible:

In [ ]:
h = get("/health")
arch = h.get("architecture")

print("model_id     :", h["model_id"])
print("architecture :", arch if arch is not None else "(not reported by this build)")
print("model_class  :", h.get("model_class", "(not reported by this build)"))

if arch == "boundary":
    print("\nGLiNER2.5 checkpoint - relation extraction is available at the model level.")
elif arch == "span":
    print("\nGLiNER2 checkpoint - no relation extraction. Swap MODEL_ID to a 2.5 model.")
else:
    print("\nThis build predates GLiNER2.5 dispatch reporting. See docs/runbook.md.")

## Where to go next

| Need | Document |
|---|---|
| Full endpoint reference and error semantics | [`api.md`](api.md) |
| Deploy, model swap, rollback, failure modes, sizing | [`runbook.md`](runbook.md) |
| What GLiNER2/2.5 is, model comparison, homelab fit | [`wiki.md`](wiki.md) |
| Jetson build decisions and the `sbsa/cu130` trap | [`../JETSON.md`](../JETSON.md) |

Live API schema, straight from the running service:

```
http://192.168.1.177:8013/docs
http://192.168.1.177:8013/redoc
http://192.168.1.177:8013/openapi.json
```